# 홀드아웃5 201~250 전체 평가 (Colab GPU)

사전등록된 홀드아웃5를 **BGE-M3 + Chroma + KOSIS API**로 끝까지 재현한다. 로컬에서 완료한 KSS/HCX 주장 구간(50개 기사, 834문장, 203개 주장)을 입력으로 사용한다.

1. Colab 메뉴에서 `런타임 → 런타임 유형 변경 → GPU`를 선택한다.
2. 이 노트북을 위에서 아래로 실행한다.
3. 업로드 셀에서 `holdout5_colab_input_bundle.zip`을 고른다.
4. API 키는 Colab Secrets의 `CLOVA_API_KEY`, `KOSIS_API_KEY`를 쓰거나 비공개 입력창에 넣는다. 키는 출력·결과 ZIP에 저장하지 않는다.
5. 마지막 셀에서 내려받은 `holdout5_gpu_results.zip`을 Codex 작업에 첨부한다.

중요: `API_ERROR=0`이 아니면 결과를 성공으로 판정하지 않는다. `probe_empty_coordinates.py`에는 반드시 `chroma_candidates.csv`를 전달한다.

In [ ]:
# 1. GPU 확인
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=True)
print(gpu.stdout.strip())
assert 'GPU' in gpu.stdout, 'GPU 런타임을 선택한 뒤 다시 실행하세요.'

In [ ]:
# 2. 무비밀 입력 번들 업로드·해제
from google.colab import files
from pathlib import Path
import io, os, shutil, subprocess, sys, zipfile

os.chdir('/content')
uploaded = files.upload()
bundle_names = [n for n in uploaded if n.endswith('.zip')]
assert len(bundle_names) == 1, 'holdout5_colab_input_bundle.zip 하나만 업로드하세요.'
bundle_bytes = uploaded[bundle_names[0]]
ROOT = Path('/content/holdout5_gpu')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as zf:
    for info in zf.infolist():
        parts = [p for p in info.filename.replace('\\', '/').split('/') if p not in {'', '.'}]
        assert '..' not in parts, f'안전하지 않은 ZIP 경로: {info.filename}'
        target = ROOT.joinpath(*parts)
        if info.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(zf.read(info))
assert (ROOT / 'data' / 'holdout5_articles.csv').exists(), '번들 해제에 실패했습니다.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'outputs' / 'holdout5_201_250'
MAP = OUT / '07_mapping_v2'
SEM = ROOT / 'data' / 'indexes' / 'kosis_bge_m3'
CHR = ROOT / 'data' / 'indexes' / 'kosis_meta_chroma_holdout5'
OUT.mkdir(parents=True, exist_ok=True)
def run(args):
    cmd = [str(x) for x in args]
    print('\n$', ' '.join(cmd[:3]), '...')
    return subprocess.run(cmd, cwd=ROOT, env=os.environ.copy(), check=True)
print('ROOT =', ROOT)

In [ ]:
# 3. 의존성 설치 (Colab 기본 CUDA PyTorch는 유지)
%pip install -q -r requirements.txt -r requirements-ml.txt
import torch, chromadb, sentence_transformers
assert torch.cuda.is_available(), 'CUDA PyTorch를 사용할 수 없습니다.'
print('torch', torch.__version__, '| cuda', torch.cuda.get_device_name(0))

In [ ]:
# 4. API 키 — 출력하거나 파일에 쓰지 않는다
from getpass import getpass
try:
    from google.colab import userdata
except Exception:
    userdata = None
def secret(name):
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            pass
    return value or getpass(f'{name}: ')
os.environ['CLOVA_API_KEY'] = secret('CLOVA_API_KEY')
os.environ['KOSIS_API_KEY'] = secret('KOSIS_API_KEY')
assert os.environ['CLOVA_API_KEY'] and os.environ['KOSIS_API_KEY']
print('API 키 2개가 메모리에 설정됐습니다 (값은 출력하지 않음).')

In [ ]:
# 5. 번들 해시와 고정 입력 검증
import csv, hashlib, json
manifest = json.loads((ROOT / 'bundle_manifest.json').read_text(encoding='utf-8-sig'))
for rel, expected in manifest['files'].items():
    actual = hashlib.sha256((ROOT / rel).read_bytes()).hexdigest()
    assert actual == expected, f'해시 불일치: {rel}'
def csv_rows(path):
    with open(path, encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))
articles = csv_rows(ROOT / 'data' / 'holdout5_articles.csv')
sentences = csv_rows(OUT / '01_sentences.csv')
contexts = csv_rows(OUT / '03_claim_contexts.csv')
assert len(articles) == 50 and len(sentences) == 834 and len(contexts) == 203
assert len({r['article_id'] for r in sentences}) == 50
assert len({r['article_id'] for r in contexts}) == 40
assert not (ROOT / '.env').exists(), '번들에 .env가 들어 있으면 중단'
print('입력 검증 완료:', len(articles), '기사 /', len(sentences), '문장 /', len(contexts), '주장')

In [ ]:
# 6. KOSIS 표 10만+건 BGE-M3 인덱스 (GPU, 체크포인트 재사용)
run([sys.executable, 'kosis_build_embedding_index.py',
     '--table-index', 'kosis_table_summary.csv',
     '--out-dir', SEM, '--batch-size', '16', '--device', 'cuda'])
assert (SEM / 'manifest.json').exists(), 'semantic manifest가 생성되지 않았습니다.'
print((SEM / 'manifest.json').read_text(encoding='utf-8')[:2000])

In [ ]:
# 7. 203개 주장에 대한 초기 표 검색·리랭킹 (GPU, 재개 가능)
run([sys.executable, 'kosis_early_retrieve.py',
     '--input', OUT / '03_claim_contexts.csv',
     '--output-candidates', OUT / '04_early_bge_candidates_top20.csv',
     '--output-context', OUT / '04_early_bge_context_top5.csv',
     '--semantic-index', SEM, '--semantic-top-k', '20',
     '--rerank-top-k', '20', '--context-top-k', '5',
     '--checkpoint-every', '10', '--device', 'cuda'])
print('초기 검색 완료')

In [ ]:
# 8. HCX-007 measurement 추출 (API, 출력 파일 기준 재개)
run([sys.executable, 'extract_hcx.py',
     '--input', OUT / '03_claim_contexts.csv',
     '--retrieval-context', OUT / '04_early_bge_context_top5.csv',
     '--output', OUT / '05_hcx_measurements.csv',
     '--model', 'HCX-007', '--sleep', '0.5'])
measurements = csv_rows(OUT / '05_hcx_measurements.csv')
print('measurement rows =', len(measurements))

In [ ]:
# 9. measurement → KOSIS 표 후보·공식 메타 (GPU + KOSIS API)
run([sys.executable, 'run_kosis_measurement_pipeline.py',
     '--input', OUT / '05_hcx_measurements.csv',
     '--table-index', ROOT / 'kosis_table_summary.csv',
     '--semantic-index', SEM, '--out-dir', MAP,
     '--retrieval-mode', 'hybrid', '--semantic-top-k', '50',
     '--rerank-top-k', '20', '--device', 'cuda'])
print('1차 mapping 산출물 =', MAP)

In [ ]:
# 10. 평가 집합 잠금
READY = MAP / '05_hcx_measurements_kosis_ready.csv'
META = MAP / '05_hcx_measurements_kosis_meta_index.csv'
TABLE_CAND = MAP / '05_hcx_measurements_kosis_table_candidates.csv'
EVAL = MAP / 'evaluation_set.csv'
run([sys.executable, 'lock_evaluation_set.py', '--ready', READY,
     '--output', EVAL, '--excluded-output', MAP / 'evaluation_set_excluded.csv',
     '--manifest', MAP / 'evaluation_set_manifest.json'])
print('evaluation measurements =', len(csv_rows(EVAL)))

In [ ]:
# 11. 홀드아웃 전용 Chroma 좌표 인덱스 (GPU)
run([sys.executable, 'kosis_build_chroma_meta_index.py',
     '--meta-index', META, '--persist-dir', CHR,
     '--collection', 'kosis_meta_coordinates',
     '--embedding-model', 'BAAI/bge-m3', '--axis-value-limit', '300',
     '--prd-se-source', TABLE_CAND, '--device', 'cuda', '--reset'])
assert (CHR / 'chroma_manifest.json').exists()
print((CHR / 'chroma_manifest.json').read_text(encoding='utf-8')[:2000])

In [ ]:
# 12. target-aware 하이브리드 좌표 검색 (GPU)
CAND = MAP / 'chroma_candidates.csv'
run([sys.executable, 'kosis_chroma_hybrid_search.py',
     '--claims', EVAL, '--table-candidates', TABLE_CAND,
     '--persist-dir', CHR, '--collection', 'kosis_meta_coordinates',
     '--output', CAND, '--stats-output', MAP / 'chroma_stats.csv',
     '--dense-top-k', '50', '--lexical-top-k', '50',
     '--rerank-top-k', '20', '--final-top-k', '10',
     '--reranker-model', 'BAAI/bge-reranker-v2-m3', '--device', 'cuda'])
print('candidate rows =', len(csv_rows(CAND)))

In [ ]:
# 13. KOSIS 좌표 검증 — API_ERROR가 0이 될 때까지 최대 5회 재시도
import time
VALID = MAP / 'chroma_validated.csv'
for attempt in range(1, 6):
    run([sys.executable, 'kosis_validate_mapping_candidates.py',
         '--input', CAND, '--meta-index', META, '--output', VALID,
         '--evaluate-all-ranks', '--strict-seeded-coordinate',
         '--item-top-k', '1', '--obj-top-k', '1',
         '--max-combinations', '1', '--allow-provisional'])
    vr = csv_rows(VALID)
    api_errors = sum(r.get('mapping_status') == 'API_ERROR' for r in vr)
    print(f'attempt={attempt} API_ERROR rows={api_errors}')
    if api_errors == 0:
        break
    time.sleep(20)
assert api_errors == 0, 'API_ERROR가 남았습니다. 잠시 뒤 이 셀을 다시 실행하세요.'

In [ ]:
# 14. 커버리지 + 빈 좌표 프로브 (후보 파일을 넘기는 것이 핵심)
COV0 = MAP / 'coverage_preprobe.csv'
PROBE = MAP / 'empty_coordinate_probe.csv'
COV = MAP / 'coverage.csv'
run([sys.executable, 'report_coverage.py', '--validated', VALID,
     '--evaluation-set', EVAL, '--article-source', READY, '--output', COV0])
run([sys.executable, 'probe_empty_coordinates.py', '--coverage', COV0,
     '--candidates', CAND, '--output', PROBE, '--periods', '20', '--delay', '0.12'])
run([sys.executable, 'report_coverage.py', '--validated', VALID,
     '--evaluation-set', EVAL, '--article-source', READY,
     '--probe', PROBE, '--output', COV])
print('coverage rows =', len(csv_rows(COV)))

In [ ]:
# 15. READY 실제값 판정
import pandas as pd
v = pd.read_csv(VALID, dtype=str, keep_default_na=False)
src = pd.read_csv(READY, dtype=str, keep_default_na=False)
ready_v = v[v['mapping_status'].eq('READY')].copy()
ready_v['_rank'] = pd.to_numeric(ready_v.get('candidate_rank', '999'), errors='coerce').fillna(999)
ready_v = ready_v.sort_values('_rank').drop_duplicates('claim_measurement_id').drop(columns='_rank')
if 'date' not in ready_v.columns and 'date' in src.columns:
    ready_v = ready_v.merge(src[['claim_measurement_id', 'date']].drop_duplicates(),
                            on='claim_measurement_id', how='left')
VERIFY_IN = MAP / 'verify_input.csv'
VERIFIED = MAP / 'verified.csv'
ready_v.to_csv(VERIFY_IN, index=False, encoding='utf-8-sig')
run([sys.executable, 'kosis_verify_claim_values.py', '--input', VERIFY_IN,
     '--output', VERIFIED, '--delay', '0.12'])
vf = pd.read_csv(VERIFIED, dtype=str, keep_default_na=False)
api_v = int(vf['verdict_code'].eq('KOSIS_API_ERROR').sum()) if 'verdict_code' in vf else 0
assert api_v == 0, '실제값 검증 API_ERROR가 있습니다. 이 셀을 다시 실행하세요.'
print(vf['verdict'].value_counts(dropna=False).to_string())

In [ ]:
# 16. 요약·결과 ZIP 다운로드 (대용량 인덱스와 API 키 제외)
from collections import Counter
coverage = csv_rows(COV)
status = Counter(r.get('mapping_status', '') for r in csv_rows(VALID))
verdicts = Counter(vf.get('verdict', pd.Series(dtype=str)).tolist())
codes = Counter(vf.get('verdict_code', pd.Series(dtype=str)).tolist())
summary = {
    'articles': 50, 'sentences': 834, 'claim_contexts': 203,
    'measurements': len(measurements),
    'evaluation_measurements': len(csv_rows(EVAL)),
    'validated_status_rows': dict(status),
    'verified_rows': len(vf), 'verdicts': dict(verdicts),
    'verdict_codes': dict(codes), 'api_error_rows': 0,
    'note': '거짓 불일치는 기사 진위와 좌표를 수동 검수한 뒤 확정한다.'
}
(MAP / 'gpu_run_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
for src_path, name in [(SEM / 'manifest.json', 'semantic_manifest.json'),
                       (CHR / 'chroma_manifest.json', 'chroma_manifest.json')]:
    shutil.copy2(src_path, MAP / name)
archive = shutil.make_archive('/content/holdout5_gpu_results', 'zip', OUT)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('다운로드:', archive)
files.download(archive)

## 완료 후

다운로드한 `holdout5_gpu_results.zip`을 원래 Codex 작업에 첨부한다. Codex가 사전등록 기준으로 API 오류, target→aggregate 거짓 확정, 불일치, 첫 50건 회귀를 최종 판정한다.